In [2]:
# Verwendete Bibliotheken
import pandas as pd
import re
import requests
from bs4 import BeautifulSoup
import time
import difflib
import statsmodels.api as sm
import statsmodels.formula.api as smf
import random
import numpy as np
from itertools import combinations
import math
from collections import Counter
import matplotlib.pyplot as plt
import seaborn as sns


In [3]:
# Liste der CSV-Dateipfade
file_paths = [
    r"C:\Users\charb\Downloads\champions-league-2017-CentralEuropeanStandardTime.csv",
    r"C:\Users\charb\Downloads\champions-league-2018-WEuropeStandardTime.csv",
    r"C:\Users\charb\Downloads\champions-league-2020-UTC.csv",
    r"C:\Users\charb\Downloads\champions-league-2021-UTC.csv",
    r"C:\Users\charb\Downloads\champions-league-2022-UTC.csv",
    r"C:\Users\charb\Downloads\champions-league-2023-UTC.csv"
]

In [4]:
# Alle CSV-Dateien einlesen und in einer Liste speichern
dfs = [pd.read_csv(fp) for fp in file_paths]

# Zusammenfügen zu einem DataFrame
combined_df = pd.concat(dfs, ignore_index=True)

In [5]:
combined_df

,Match Number,Round Number,Date,Location,Home Team,Away Team,Group,Result
0,1,1,12/09/2017 20:45,"Estádio da Luz, Lisbon",Benfica,CSKA Moscow,Group A,1 - 2
1,2,1,12/09/2017 20:45,"Old Trafford, Manchester",Manchester United,Basel,Group A,3 - 0
2,13,1,12/09/2017 20:45,"Allianz Arena, Munich",Bayern Munich,Anderlecht,Group B,3 - 0
3,14,1,12/09/2017 20:45,"Celtic Park, Glasgow",Celtic,Paris Saint-Germain,Group B,0 - 5
4,25,1,12/09/2017 20:45,"Stamford Bridge, London",Chelsea,Qarabag,Group C,6 - 0
...,...,...,...,...,...,...,...,...
745,121,SF Game 1,30/04/2024 19:00,Fußball Arena München,Bayern,Real Madrid,NaN,2 - 2
746,122,SF Game 1,01/05/2024 19:00,BVB Stadion Dortmund,Dortmund,Paris,NaN,1 - 0
747,123,SF Game 2,07/05/2024 19:00,Parc des Princes,Paris,Dortmund,NaN,0 - 1
748,124,SF Game 2,08/05/2024 19:00,Estadio Santiago Bernabéu,Real Madrid,Bayern,NaN,2 - 1


In [6]:
# Funktion zur Aufteilung des Spielergebnisses in Heim- und Auswärtstore
def split_result(result):
    try:
        home, away = result.strip().split('-')
        return int(home.strip()), int(away.strip())
    except:
        return None, None

# Ergebnis-Spalte anwenden und in zwei neue Spalten aufteilen
combined_df[['Home Goals', 'Away Goals']] = combined_df['Result'].apply(
    lambda x: pd.Series(split_result(x))
)

In [7]:
combined_df

,Match Number,Round Number,Date,Location,Home Team,Away Team,Group,Result,Home Goals,Away Goals
0,1,1,12/09/2017 20:45,"Estádio da Luz, Lisbon",Benfica,CSKA Moscow,Group A,1 - 2,1,2
1,2,1,12/09/2017 20:45,"Old Trafford, Manchester",Manchester United,Basel,Group A,3 - 0,3,0
2,13,1,12/09/2017 20:45,"Allianz Arena, Munich",Bayern Munich,Anderlecht,Group B,3 - 0,3,0
3,14,1,12/09/2017 20:45,"Celtic Park, Glasgow",Celtic,Paris Saint-Germain,Group B,0 - 5,0,5
4,25,1,12/09/2017 20:45,"Stamford Bridge, London",Chelsea,Qarabag,Group C,6 - 0,6,0
...,...,...,...,...,...,...,...,...,...,...
745,121,SF Game 1,30/04/2024 19:00,Fußball Arena München,Bayern,Real Madrid,NaN,2 - 2,2,2
746,122,SF Game 1,01/05/2024 19:00,BVB Stadion Dortmund,Dortmund,Paris,NaN,1 - 0,1,0
747,123,SF Game 2,07/05/2024 19:00,Parc des Princes,Paris,Dortmund,NaN,0 - 1,0,1
748,124,SF Game 2,08/05/2024 19:00,Estadio Santiago Bernabéu,Real Madrid,Bayern,NaN,2 - 1,2,1


In [8]:
# ############################################## 2017-18 ##############################################

# # 1. Champions-League-Spiele laden
# df = pd.read_csv("champions-league-2017-CentralEuropeanStandardTime.csv")
# df["Date"] = pd.to_datetime(df["Date"], dayfirst=True)
# df_sorted = df.sort_values(by="Date")

# # 2. Früheste Spiele je Runde extrahieren
# first_dates_per_round = (
#     df_sorted.groupby("Round Number")["Date"]
#     .min()
#     .drop_duplicates()
#     .sort_values()
# )
# date_list_for_urls = first_dates_per_round.dt.strftime("%Y-%m-%d").tolist()

# # 3. Tabellen von clubelo.com scrapen
# headers = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64)"}
# ucl_rankings = {}

# for date in date_list_for_urls:
#     url = f"http://clubelo.com/{date}/UCL/Ranking"
#     print(f"Scraping: {url}")
    
#     try:
#         response = requests.get(url, headers=headers)
#         soup = BeautifulSoup(response.text, "html.parser")
#         table = soup.find("table", class_="ranking")
        
#         if table:
#             df_table = pd.read_html(str(table))[0]
#             df_table["Date"] = date
#             ucl_rankings[date] = df_table
#         else:
#             print(f"Tabelle nicht gefunden für {date}")
#             ucl_rankings[date] = None
#     except Exception as e:
#         print(f"Fehler bei {date}: {e}")
#         ucl_rankings[date] = None
    
#     time.sleep(5)

# # 4. Alle gescrapten Tabellen kombinieren
# all_tables = [df for df in ucl_rankings.values() if df is not None]
# df_combined = pd.concat(all_tables, ignore_index=True)
# df_combined.to_csv("ucl_elo_rankings_2017_2018.csv", index=False)
# print("Fertig! Tabellen gespeichert in 'ucl_elo_rankings_2017_2018.csv'")

In [9]:
df_2017_18 = pd.read_csv("ucl_elo_rankings_2017_2018.csv")

In [10]:
df_2017_18.head(50)

,Rank,Club,Elo,Unnamed: 3,Last Match,Coach,Date
0,1,1 Real Madrid,2072,NaN,2017-09-09,Zinédine Zidane (since 2016-01-05),2017-09-12
1,2,2 Barcelona,2033,NaN,2017-09-09,Ernesto Valverde (since 2017-07-01),2017-09-12
2,3,3 Bayern,1974,NaN,2017-09-09,Carlo Ancelotti (since 2016-07-01),2017-09-12
3,4,4 Juventus,1954,NaN,2017-09-09,Massimiliano Allegri (since 2014-07-16),2017-09-12
4,5,5 Atlético,1951,NaN,2017-09-09,Diego Simeone (since 2011-12-23),2017-09-12
5,6,6 Chelsea,1919,NaN,2017-09-09,Antonio Conte (since 2016-07-01),2017-09-12
6,7,7 Tottenham,1893,NaN,2017-09-09,Mauricio Pochettino (since 2014-07-01),2017-09-12
7,8,8 Paris SG,1889,NaN,2017-09-08,Unai Emery (since 2016-07-01),2017-09-12
8,9,9 Man City,1889,NaN,2017-09-09,Pep Guardiola (since 2016-07-01),2017-09-12
9,10,10 Napoli,1889,NaN,2017-09-10,Maurizio Sarri (since 2015-06-11),2017-09-12


In [11]:
# ############################################## 2018-19 ##############################################

# # 1. CSV-Datei mit CL-Spielen laden
# df = pd.read_csv("champions-league-2018-WEuropeStandardTime.csv")
# df["Date"] = pd.to_datetime(df["Date"], dayfirst=True)
# df_sorted = df.sort_values(by="Date")

# # 2. Frühestes Spiel je Runde extrahieren (für URLs)
# first_dates_per_round = (
#     df_sorted.groupby("Round Number")["Date"]
#     .min()
#     .drop_duplicates()
#     .sort_values()
# )
# date_list_for_urls = first_dates_per_round.dt.strftime("%Y-%m-%d").tolist()

# # 3. Rankings von clubelo.com scrapen
# headers = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64)"}
# ucl_rankings = {}

# for date in date_list_for_urls:
#     url = f"http://clubelo.com/{date}/UCL/Ranking"
#     print(f"Scraping: {url}")
    
#     try:
#         response = requests.get(url, headers=headers)
#         soup = BeautifulSoup(response.text, "html.parser")
#         table = soup.find("table", class_="ranking")
        
#         if table:
#             df_table = pd.read_html(str(table))[0]
#             df_table["Date"] = date
#             ucl_rankings[date] = df_table
#         else:
#             print(f"Tabelle nicht gefunden für {date}")
#             ucl_rankings[date] = None
#     except Exception as e:
#         print(f"Fehler bei {date}: {e}")
#         ucl_rankings[date] = None

#     time.sleep(5)

# # 4. Alle Tabellen zusammenfügen und speichern
# all_tables = [df for df in ucl_rankings.values() if df is not None]
# df_combined = pd.concat(all_tables, ignore_index=True)
# df_combined.to_csv("ucl_elo_rankings_2018_2019.csv", index=False)
# print("Fertig! Tabellen gespeichert in 'ucl_elo_rankings_2018_2019.csv'")

In [12]:
df_2018_19 = pd.read_csv("ucl_elo_rankings_2018_2019.csv")

In [13]:
df_2018_19

,Rank,Club,Elo,Unnamed: 3,Last Match,Coach,Date
0,1,1 Barcelona,2036,NaN,2018-09-15,Ernesto Valverde (since 2017-07-01),2018-09-18
1,2,2 Real Madrid,2024,NaN,2018-09-15,Julen Lopetegui (since 2018-07-01),2018-09-18
2,3,3 Man City,1977,NaN,2018-09-15,Pep Guardiola (since 2016-07-01),2018-09-18
3,4,4 Juventus,1966,NaN,2018-09-16,Massimiliano Allegri (since 2014-07-16),2018-09-18
4,5,5 Bayern,1960,NaN,2018-09-15,Niko Kovač (since 2018-07-01),2018-09-18
...,...,...,...,...,...,...,...
245,2,3 Liverpool,2003,NaN,2019-05-04,Jürgen Klopp (since 2015-10-08),2019-05-07
246,3,8 Tottenham,1877,NaN,2019-05-04,Mauricio Pochettino (since 2014-07-01),2019-05-07
247,4,13 Ajax,1836,NaN,2019-04-30,Erik ten Hag (since 2018-01-01),2019-05-07
248,1,1 Liverpool,2044,NaN,2019-05-12,Jürgen Klopp (since 2015-10-08),2019-06-01


In [14]:
# ############################################## 2020-21 ##############################################

# # 1. CL-Spieldaten einlesen und nach Datum sortieren
# df = pd.read_csv("champions-league-2020-UTC.csv")
# df["Date"] = pd.to_datetime(df["Date"], dayfirst=True)
# df_sorted = df.sort_values(by="Date")

# # 2. Frühestes Spiel je Runde als Grundlage für URL-Erstellung
# first_dates_per_round = (
#     df_sorted.groupby("Round Number")["Date"]
#     .min()
#     .drop_duplicates()
#     .sort_values()
# )
# date_list_for_urls = first_dates_per_round.dt.strftime("%Y-%m-%d").tolist()

# # 3. Elo-Rankings von clubelo.com scrapen
# headers = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64)"}
# ucl_rankings = {}

# for date in date_list_for_urls:
#     url = f"http://clubelo.com/{date}/UCL/Ranking"
#     print(f"Scraping: {url}")
    
#     try:
#         response = requests.get(url, headers=headers)
#         soup = BeautifulSoup(response.text, "html.parser")
#         table = soup.find("table", class_="ranking")
        
#         if table:
#             df_table = pd.read_html(str(table))[0]
#             df_table["Date"] = date
#             ucl_rankings[date] = df_table
#         else:
#             print(f"Tabelle nicht gefunden für {date}")
#             ucl_rankings[date] = None
#     except Exception as e:
#         print(f"Fehler bei {date}: {e}")
#         ucl_rankings[date] = None
        
#     time.sleep(5)

# # 4. Tabellen kombinieren und als CSV speichern
# all_tables = [df for df in ucl_rankings.values() if df is not None]
# df_combined = pd.concat(all_tables, ignore_index=True)
# df_combined.to_csv("ucl_elo_rankings_2020_2021.csv", index=False)
# print("Fertig! Tabellen gespeichert in 'ucl_elo_rankings_2020_2021.csv'")

In [15]:
df_2020_21 = pd.read_csv("ucl_elo_rankings_2020_2021.csv")

In [16]:
df_2020_21

,Rank,Club,Elo,Unnamed: 3,Last Match,Coach,Date
0,1,1 Bayern,2025,NaN,2020-10-17,Hansi Flick (since 2019-11-03),2020-10-20
1,2,2 Liverpool,1966,NaN,2020-10-17,Jürgen Klopp (since 2015-10-08),2020-10-20
2,3,3 Barcelona,1953,NaN,2020-10-17,Ronald Koeman (since 2020-08-19),2020-10-20
3,4,4 Man City,1950,NaN,2020-10-17,Pep Guardiola (since 2016-07-01),2020-10-20
4,5,5 Paris SG,1916,NaN,2020-10-16,Thomas Tuchel (since 2018-07-01),2020-10-20
...,...,...,...,...,...,...,...
245,2,4 Real Madrid,1943,NaN,2021-05-01,Zinédine Zidane (since 2019-03-11),2021-05-04
246,3,7 Chelsea,1900,NaN,2021-05-01,Thomas Tuchel (since 2021-01-26),2021-05-04
247,4,10 Paris SG,1881,NaN,2021-05-01,Mauricio Pochettino (since 2021-01-02),2021-05-04
248,1,1 Man City,2024,NaN,2021-05-23,Pep Guardiola (since 2016-07-01),2021-05-29


In [17]:
# ############################################## 2021-22 ##############################################

# # 1. CL-Spieldaten einlesen und nach Datum sortieren
# df = pd.read_csv("champions-league-2021-UTC.csv")
# df["Date"] = pd.to_datetime(df["Date"], dayfirst=True)
# df_sorted = df.sort_values(by="Date")

# # 2. Pro Runde das früheste Spieldatum extrahieren (für die URLs)
# first_dates_per_round = (
#     df_sorted.groupby("Round Number")["Date"]
#     .min()
#     .drop_duplicates()
#     .sort_values()
# )
# date_list_for_urls = first_dates_per_round.dt.strftime("%Y-%m-%d").tolist()

# # 3. Elo-Rankings zu den Spieltagen scrapen
# headers = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64)"}
# ucl_rankings = {}

# for date in date_list_for_urls:
#     url = f"http://clubelo.com/{date}/UCL/Ranking"
#     print(f"Scraping: {url}")
    
#     try:
#         response = requests.get(url, headers=headers)
#         soup = BeautifulSoup(response.text, "html.parser")
#         table = soup.find("table", class_="ranking")
        
#         if table:
#             df_table = pd.read_html(str(table))[0]
#             df_table["Date"] = date
#             ucl_rankings[date] = df_table
#         else:
#             print(f"Tabelle nicht gefunden für {date}")
#             ucl_rankings[date] = None
#     except Exception as e:
#         print(f"Fehler bei {date}: {e}")
#         ucl_rankings[date] = None
        
#     time.sleep(5)

# # 4. Alle vorhandenen Rankings zusammenfügen und speichern
# all_tables = [df for df in ucl_rankings.values() if df is not None]
# df_combined = pd.concat(all_tables, ignore_index=True)
# df_combined.to_csv("ucl_elo_rankings_2021_2022.csv", index=False)
# print("Fertig! Tabellen gespeichert in 'ucl_elo_rankings_2021_2022.csv'")

In [18]:
df_2021_22 = pd.read_csv("ucl_elo_rankings_2021_2022.csv")

In [19]:
df_2021_22.head(6)

,Rank,Club,Elo,Unnamed: 3,Last Match,Coach,Date
0,1,1 Man City,2016,NaN,2021-09-11,Pep Guardiola (since 2016-07-01),2021-09-14
1,2,2 Bayern,2000,NaN,2021-09-11,Julian Nagelsmann (since 2021-07-01),2021-09-14
2,3,3 Liverpool,1955,NaN,2021-09-12,Jürgen Klopp (since 2015-10-08),2021-09-14
3,4,4 Real Madrid,1945,NaN,2021-09-12,Carlo Ancelotti (since 2021-07-01),2021-09-14
4,5,5 Chelsea,1944,NaN,2021-09-11,Thomas Tuchel (since 2021-01-26),2021-09-14
5,6,6 Man United,1937,NaN,2021-09-11,Ole Gunnar Solskjær (since 2018-12-19),2021-09-14


In [20]:
# ############################################## 2022-23 ##############################################

# # 1. CL-Spieldaten einlesen und nach Datum sortieren
# df = pd.read_csv("champions-league-2022-UTC.csv")
# df["Date"] = pd.to_datetime(df["Date"], dayfirst=True)
# df_sorted = df.sort_values(by="Date")

# # 2. Pro Runde das früheste Spieldatum extrahieren (für die URLs)
# first_dates_per_round = (
#     df_sorted.groupby("Round Number")["Date"]
#     .min()
#     .drop_duplicates()
#     .sort_values()
# )
# date_list_for_urls = first_dates_per_round.dt.strftime("%Y-%m-%d").tolist()

# # 3. Elo-Rankings zu den Spieltagen scrapen
# headers = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64)"}
# ucl_rankings = {}

# for date in date_list_for_urls:
#     url = f"http://clubelo.com/{date}/UCL/Ranking"
#     print(f"Scraping: {url}")
    
#     try:
#         response = requests.get(url, headers=headers)
#         soup = BeautifulSoup(response.text, "html.parser")
#         table = soup.find("table", class_="ranking")
        
#         if table:
#             df_table = pd.read_html(str(table))[0]
#             df_table["Date"] = date
#             ucl_rankings[date] = df_table
#         else:
#             print(f"Tabelle nicht gefunden für {date}")
#             ucl_rankings[date] = None
#     except Exception as e:
#         print(f"Fehler bei {date}: {e}")
#         ucl_rankings[date] = None
        
#     time.sleep(5)

# # 4. Alle vorhandenen Rankings zusammenfügen und speichern
# all_tables = [df for df in ucl_rankings.values() if df is not None]
# df_combined = pd.concat(all_tables, ignore_index=True)
# df_combined.to_csv("ucl_elo_rankings_2022_2023.csv", index=False)
# print("Fertig! Tabellen gespeichert in 'ucl_elo_rankings_2022_2023.csv'")

In [21]:
df_2022_23 = pd.read_csv("ucl_elo_rankings_2022_2023.csv")

In [22]:
df_2022_23

,Rank,Club,Elo,Unnamed: 3,Last Match,Coach,Date
0,1,1 Man City,2025,NaN,2022-09-03,Pep Guardiola (since 2016-07-01),2022-09-06
1,2,2 Liverpool,2007,NaN,2022-09-03,Jürgen Klopp (since 2015-10-08),2022-09-06
2,3,3 Real Madrid,1996,NaN,2022-09-03,Carlo Ancelotti (since 2021-07-01),2022-09-06
3,4,4 Bayern,1962,NaN,2022-09-03,Julian Nagelsmann (since 2021-07-01),2022-09-06
4,5,5 Barcelona,1898,NaN,2022-09-03,Xavi (since 2021-11-08),2022-09-06
...,...,...,...,...,...,...,...
245,2,4 Real Madrid,1926,NaN,2023-05-13,Carlo Ancelotti (since 2021-07-01),2023-05-16
246,3,8 Inter,1884,NaN,2023-05-13,Simone Inzaghi (since 2021-07-01),2023-05-16
247,4,17 Milan,1820,NaN,2023-05-13,Stefano Pioli (since 2019-10-09),2023-05-16
248,1,1 Man City,2072,NaN,2023-05-28,Pep Guardiola (since 2016-07-01),2023-06-10


In [23]:
# ############################################## 2023-24 ##############################################

# # 1. Spieldaten einlesen und nach Datum sortieren
# df = pd.read_csv("champions-league-2023-UTC.csv")
# df["Date"] = pd.to_datetime(df["Date"], dayfirst=True)
# df_sorted = df.sort_values(by="Date")

# # 2. Früheste Spieltermine pro Runde als Datums-Liste für den Abruf vorbereiten
# first_dates_per_round = (
#     df_sorted.groupby("Round Number")["Date"]
#     .min()
#     .drop_duplicates()
#     .sort_values()
# )
# date_list_for_urls = first_dates_per_round.dt.strftime("%Y-%m-%d").tolist()

# # 3. Abruf der ClubElo-Rankings zu den Spieltagen
# headers = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64)"}
# ucl_rankings = {}

# for date in date_list_for_urls:
#     url = f"http://clubelo.com/{date}/UCL/Ranking"
#     print(f"Scraping: {url}")
    
#     try:
#         response = requests.get(url, headers=headers)
#         soup = BeautifulSoup(response.text, "html.parser")
#         table = soup.find("table", class_="ranking")
        
#         if table:
#             df_table = pd.read_html(str(table))[0]
#             df_table["Date"] = date
#             ucl_rankings[date] = df_table
#         else:
#             print(f"Tabelle nicht gefunden für {date}")
#             ucl_rankings[date] = None
#     except Exception as e:
#         print(f"Fehler bei {date}: {e}")
#         ucl_rankings[date] = None
        
#     time.sleep(5)

# # 4. Tabellen zusammenfügen und lokal speichern
# all_tables = [df for df in ucl_rankings.values() if df is not None]
# df_combined = pd.concat(all_tables, ignore_index=True)
# df_combined.to_csv("ucl_elo_rankings_2023_2024.csv", index=False)
# print("Fertig! Tabellen gespeichert in 'ucl_elo_rankings_2023_2024.csv'")

In [24]:
df_2023_24 = pd.read_csv("ucl_elo_rankings_2023_2024.csv")

In [25]:
df_2023_24.head(5)

,Rank,Club,Elo,Unnamed: 3,Last Match,Coach,Date
0,1,1 Man City,2091,NaN,2023-09-16,Pep Guardiola (since 2016-07-01),2023-09-19
1,2,3 Bayern,1940,NaN,2023-09-15,Thomas Tuchel (since 2023-03-24),2023-09-19
2,3,4 Arsenal,1932,NaN,2023-09-17,Mikel Arteta (since 2019-12-22),2023-09-19
3,4,5 Real Madrid,1922,NaN,2023-09-17,Carlo Ancelotti (since 2021-07-01),2023-09-19
4,5,6 Inter,1909,NaN,2023-09-16,Simone Inzaghi (since 2021-07-01),2023-09-19


In [26]:
# Einlesen des vorbereiteten kombinierten Champions-League-Datensatzes (aus Excel-Datei)
combined_df = pd.read_excel(r"C:\Users\charb\Downloads\combined_df_full.xlsx")

In [27]:
combined_df

,Match Number,Round Number,Date,Location,Home Team,Away Team,Group,Result,Home Goals,Away Goals,Elo Home,Elo Away
0,1,1,2017-09-12 20:45:00,"Estádio da Luz, Lisbon",Benfica,CSKA Moscow,Group A,1 - 2,1,2,1797.0,1723.0
1,2,1,2017-09-12 20:45:00,"Old Trafford, Manchester",Manchester United,Basel,Group A,3 - 0,3,0,NaN,1621.0
2,13,1,2017-09-12 20:45:00,"Allianz Arena, Munich",Bayern Munich,Anderlecht,Group B,3 - 0,3,0,NaN,1656.0
3,14,1,2017-09-12 20:45:00,"Celtic Park, Glasgow",Celtic,Paris Saint-Germain,Group B,0 - 5,0,5,1615.0,NaN
4,25,1,2017-09-12 20:45:00,"Stamford Bridge, London",Chelsea,Qarabag,Group C,6 - 0,6,0,1919.0,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...
745,121,SF Game 1,2024-04-30 19:00:00,Fußball Arena München,Bayern Munich,Real Madrid,NaN,2 - 2,2,2,1941.0,1984.0
746,122,SF Game 1,2024-05-01 19:00:00,BVB Stadion Dortmund,Borussia Dortmund,Paris Saint-Germain,NaN,1 - 0,1,0,1880.0,NaN
747,123,SF Game 2,2024-05-07 19:00:00,Parc des Princes,Paris Saint-Germain,Borussia Dortmund,NaN,0 - 1,0,1,NaN,1890.0
748,124,SF Game 2,2024-05-08 19:00:00,Estadio Santiago Bernabéu,Real Madrid,Bayern Munich,NaN,2 - 1,2,1,1988.0,1927.0


In [28]:
# Laden der zusammengeführten historischen Elo-Ratings für alle Champions-League-Saisons (2017–2024)
# Diese Datei wurde zuvor aus den ClubElo-Webseiten gescraped und in Excel gespeichert
elo_data_merged = pd.read_excel(r"C:\Users\charb\Downloads\elo_data_merged_2017_2024.xlsx")

In [29]:
elo_data_merged

,Rank,Club,Elo,Unnamed: 3,Last Match,Coach,Date,Clean Club
0,1,1 Real Madrid,2072,NaN,2017-09-09,Zinédine Zidane (since 2016-01-05),2017-09-12,real madrid
1,2,2 Barcelona,2033,NaN,2017-09-09,Ernesto Valverde (since 2017-07-01),2017-09-12,barcelona
2,3,3 Bayern Munich,1974,NaN,2017-09-09,Carlo Ancelotti (since 2016-07-01),2017-09-12,Bayern Munich
3,4,4 Juventus,1954,NaN,2017-09-09,Massimiliano Allegri (since 2014-07-16),2017-09-12,juventus
4,5,5 Atlético Madrid,1951,NaN,2017-09-09,Diego Simeone (since 2011-12-23),2017-09-12,Atlético Madrid
...,...,...,...,...,...,...,...,...
1495,2,6 Bayern Munich,1927,NaN,2024-05-04,Thomas Tuchel (since 2023-03-24),2024-05-07,Bayern Munich
1496,3,7 Paris Saint-Germain,1910,NaN,2024-05-01,Luis Enrique (since 2023-07-05),2024-05-07,Paris Saint-Germain
1497,4,9 Borussia Dortmund,1890,NaN,2024-05-04,Edin Terzić (since 2022-05-24),2024-05-07,Borussia Dortmund
1498,1,2 Real Madrid,1988,NaN,2024-05-25,Carlo Ancelotti (since 2021-07-01),2024-06-01,real madrid


In [30]:
# Datumsspalten in beide DataFrames konvertieren (für zeitbasiertes Matching)
elo_data_merged["Date"] = pd.to_datetime(elo_data_merged["Date"])
combined_df["Date"] = pd.to_datetime(combined_df["Date"], dayfirst=True)

# Clubnamen bereinigen (z. B. '1 Man City' → 'man city') für einheitlichen Abgleich
def clean_club_name(name):
    name = str(name)
    name = re.sub(r"^\d+\s+", "", name)
    return name.strip().lower()

# Neue Spalte 'Clean Club' im Elo-Datensatz zur Vereinfachung des Abgleichs
elo_data_merged["Clean Club"] = elo_data_merged["Club"].apply(clean_club_name)

# Zwei neue Spalten zur Speicherung der passenden Elo-Werte im Spiel-Datensatz
combined_df["Elo Home"] = None
combined_df["Elo Away"] = None

# Hauptschleife: Für jedes Spiel das passende Elo-Rating der Teams ermitteln
for idx, row in combined_df.iterrows():
    match_date = row["Date"]
    home_team = clean_club_name(row["Home Team"])
    away_team = clean_club_name(row["Away Team"])

    # Letztes verfügbares Elo-Datum vor oder am Spieltag
    elo_date = elo_data_merged[elo_data_merged["Date"] <= match_date]["Date"].max()

    if pd.isna(elo_date):
        continue

    snapshot = elo_data_merged[elo_data_merged["Date"] == elo_date]

    # Elo-Rating für Heimteam zuweisen
    home_row = snapshot[snapshot["Clean Club"] == home_team]
    if not home_row.empty:
        combined_df.at[idx, "Elo Home"] = home_row.iloc[0]["Elo"]

    # Elo-Rating für Auswärtsteam zuweisen
    away_row = snapshot[snapshot["Clean Club"] == away_team]
    if not away_row.empty:
        combined_df.at[idx, "Elo Away"] = away_row.iloc[0]["Elo"]


In [31]:
# Ergebnis anzeigen
combined_df.head(50)

,Match Number,Round Number,Date,Location,Home Team,Away Team,Group,Result,Home Goals,Away Goals,Elo Home,Elo Away
0,1,1,2017-09-12 20:45:00,"Estádio da Luz, Lisbon",Benfica,CSKA Moscow,Group A,1 - 2,1,2,1797,1723
1,2,1,2017-09-12 20:45:00,"Old Trafford, Manchester",Manchester United,Basel,Group A,3 - 0,3,0,1879,1621
2,13,1,2017-09-12 20:45:00,"Allianz Arena, Munich",Bayern Munich,Anderlecht,Group B,3 - 0,3,0,1974,1656
3,14,1,2017-09-12 20:45:00,"Celtic Park, Glasgow",Celtic,Paris Saint-Germain,Group B,0 - 5,0,5,1615,1889
4,25,1,2017-09-12 20:45:00,"Stamford Bridge, London",Chelsea,Qarabag,Group C,6 - 0,6,0,1919,1511
5,26,1,2017-09-12 20:45:00,"Stadio Olimpico, Rome",Roma,Atlético Madrid,Group C,0 - 0,0,0,1832,1951
6,37,1,2017-09-12 20:45:00,"Camp Nou, Barcelona",Barcelona,Juventus,Group D,3 - 0,3,0,2033,1954
7,38,1,2017-09-12 20:45:00,"Karaiskakis Stadium, Piraeus",Olympiacos,Sporting CP,Group D,2 - 3,2,3,1660,1720
8,49,1,2017-09-13 20:45:00,"Ljudski vrt, Maribor",Maribor,Spartak Moscow,Group E,1 - 1,1,1,1434,1659
9,50,1,2017-09-13 20:45:00,"Anfield, Liverpool",Liverpool,Sevilla,Group E,2 - 2,2,2,1852,1829


In [32]:
# Zeige alle Spiele, bei denen der Elo-Wert für Heim- oder Auswärtsteam fehlt
missing_elos = combined_df[
    combined_df["Elo Home"].isna() | combined_df["Elo Away"].isna()
]

# Anzahl dieser Zeilen anzeigen
print(f"Es gibt {missing_elos.shape[0]} Zeilen mit fehlenden Elo-Werten")

Es gibt 0 Zeilen mit fehlenden Elo-Werten


In [33]:
# Zeilenweise Verarbeitung
expanded_rows = []

for idx, row in combined_df.iterrows():
    if pd.isna(row["Elo Home"]) or pd.isna(row["Elo Away"]):
        continue  # Spiele ohne vollständige Elo-Werte überspringen

    # Elo-Differenzen aus Sicht beider Teams
    elo_diff_home = row["Elo Home"] - row["Elo Away"]
    elo_diff_away = row["Elo Away"] - row["Elo Home"]

    # Heimvorteil nur setzen, wenn kein Finale
    is_final = str(row["Round Number"]).strip().lower() == "final"
    home_dummy = 0 if is_final else 1
    away_dummy = 0  # Auswärtsteam hat nie Heimvorteil

    # Heimteam-Zeile hinzufügen
    expanded_rows.append({
        "Date": row["Date"],
        "Team": row["Home Team"],
        "Goals": row["Home Goals"],
        "Elo_Diff": elo_diff_home,
        "Home": home_dummy
    })

    # Auswärtsteam-Zeile hinzufügen
    expanded_rows.append({
        "Date": row["Date"],
        "Team": row["Away Team"],
        "Goals": row["Away Goals"],
        "Elo_Diff": elo_diff_away,
        "Home": away_dummy
    })

# In DataFrame umwandeln
model_df = pd.DataFrame(expanded_rows)

# Vorschau der neuen Struktur
model_df.head()


,Date,Team,Goals,Elo_Diff,Home
0,2017-09-12 20:45:00,Benfica,1,74,1
1,2017-09-12 20:45:00,CSKA Moscow,2,-74,0
2,2017-09-12 20:45:00,Manchester United,3,258,1
3,2017-09-12 20:45:00,Basel,0,-258,0
4,2017-09-12 20:45:00,Bayern Munich,3,318,1


In [34]:
model_df.to_excel("model_df.xlsx", index=False)